# Introduction

This notebook demonstrates the general workflow to process the cleaned parquets from generated from `data_cleaning.ipynb`.

# Imports

In [ ]:
from pathlib import Path
import time

from IPython.display import display
from ipywidgets import interact, interactive, interactive_output
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit
import scipy.signal as signal
import tomli

# Functions

## Plotting

In [ ]:
def plot_signal(
    data: pd.DataFrame, 
    sample_interval: float = 2
) -> tuple[plt.Figure, plt.Axes]:
    if data is None: return None
    
    fig, ax = plt.subplots(figsize=(15,8))
    ax.plot(
        np.arange(0, data.shape[0] * sample_interval, sample_interval), 
        data
    )

    ax.grid(visible=True)
    ax.tick_params(axis='both', labelsize=14)
    
    ax.set_xlabel("time (ns)", fontsize=18)
    ax.set_ylabel("amplitude ($V$)", fontsize=18)
    
    return fig, ax


def plot_bases(
    signal: pd.Series, 
    peak_idx: int,
    peak_height: float,
    left_idx: int,
    left_idx_user: int,
    right_idx: int,
    right_idx_user: int,
    sample_interval: float = 2
) -> tuple[plt.Figure, plt.Axes]:
    
    if signal is None:
        return None
    
    fig, ax = plot_signal(signal, sample_interval)

    # Plot Peak
    ax.plot(
        peak_idx * sample_interval, 
        peak_height, 
        'rx'
    )
    
    # Left Base
    ax.plot(
        left_idx * sample_interval,
        signal[left_idx] - 0.02,
        'r^',
        label="Left Base (SciPy)"
    )
    
    
    ax.plot(
        left_idx_user * sample_interval,
        signal[left_idx_user] + 0.02,
        'gv',
        label="Left Base (User)"
    )
    
    # Right Base
    ax.plot(
        right_idx * sample_interval,
        signal[right_idx] - 0.02,
        'r^',
        label="Right Base (SciPy)"
    )
    
    ax.plot(
        right_idx_user * sample_interval,
        signal[right_idx_user] + 0.02,
        'gv',
        label="Right Base (User)"
    )
    
    ax.set_title(f"Plot of Signal {signal.name}")
    ax.legend()
    
    return fig, ax

## PSD

In [ ]:
def generate_psd(
    df: pd.DataFrame, left_bases: dict, right_bases: dict, amplitudes: dict
) -> pd.DataFrame:
    def process(series: pd.Series) -> dict:
        q_total = series[left_bases[series.name]:].sum()
        q_tail = series[right_bases[series.name]:].sum()
        q_peak = q_total - q_tail
        
        res = {
            "q_total": q_total,
            "q_tail": q_tail,
            "q_peak": q_peak,
            "q_tail_peak": q_tail / q_peak,
            "q_tail_total": q_tail / q_total
        }

        return res
    
    psd_report = pd.DataFrame({signal_id: process(df[signal_id]) for signal_id in df.columns})
    psd_report.loc["amplitude"] = amplitudes
    return psd_report


def df_to_psd(
    df: pd.DataFrame, 
    dc_offset: float,
    peak_offset, 
    tail_onset,
) -> pd.DataFrame:
    """
    This function applies the full workflow of this notebook to the input df.
    
    Args
    -----
    peak_offset: index shifted from the peak where its base may begin; used to calculate the peak's base value
    tail_onset: index shifted from the peak where the tail should start 
    """
    df = df - dc_offset
    # df_norm = (df - df.min()) / (df.max() - df.min())
    output = df.apply(lambda x: get_bases(x, peak_idx[x.name][0], peak_offset, tail_onset))
    left_bases, right_bases = output.iloc[0,:], output.iloc[1,:]
    return generate_psd(df, left_bases, right_bases, df.max())

## Peak Finding

In [ ]:
def get_bases(series: pd.Series, peak_idx: int, peak_offset: int=10, tail_offset=5):
    return get_left_bases(series, peak_idx, peak_offset), get_right_bases(series, peak_idx, tail_offset)


def get_left_bases(series: pd.Series, peak_idx: int, peak_offset: int=0):
    return series[peak_idx - peak_offset: peak_idx].idxmin()


def get_right_bases(series: pd.Series, peak_idx: int, tail_offset: int=0):
    # return series[peak_idx: peak_idx + tail_onset].idxmin()
    return peak_idx + tail_offset

## FOM

In [ ]:
def gaussian(x, mu, sigma, A):
    return A * np.exp(-((x - mu)**2 / (2 * (sigma**2))))

def bimodal(x, mu1, sigma1, A1, mu2, sigma2, A2):
    return gaussian(x, mu1, sigma1, A1) + gaussian(x, mu2, sigma2, A2)

def FWHM(sigma):
    return 2 * np.sqrt(2 * np.log(2)) * sigma

def FOM(mu1, sigma1, mu2, sigma2):
    return abs(mu2 - mu1) / (FWHM(sigma1) + FWHM(sigma2))

def plot_fom(
    psd: list, 
    n_bins: int,
    mu1: float, 
    sigma1: float, 
    A1: float, 
    mu2: float, 
    sigma2: float, 
    A2: float,
    unimodal: bool = False
) -> tuple[plt.Figure, plt.Axes]:    
    counts, bins = np.histogram(psd, n_bins)
    starting_guesses = (mu1, sigma1, A1, mu2, sigma2, A2)
    params, _cov = curve_fit(bimodal, bins[:-1], counts, starting_guesses)
    
    fig, ax = plt.subplots(figsize=(12,9))
    
    ax.plot(
        bins[:-1],
        gaussian(bins[:-1], *params[0:3]),
        label="Fitted PSD",
    )

    ax.plot(
        bins[:-1],
        counts,
        "r--",
        label="Original PSD"
    )

    fom_val = "N/A" if unimodal else f"FOM(mu1, sigma1, mu2, sigma2):.3f"
    ax.set_xlim(-0.1, 0.4)
    ax.set_title(f"FoM: {fom_val}")
    ax.legend()

    ax.set_ylabel("counts", fontsize=14)
    ax.set_xlabel("tail/total (a.u.)", fontsize=14)
    
    return fig, ax

# Pre-Processing

In [ ]:
EXP_ROOT = Path("../sample_datasets/20220824_CERC_background/processed_data/cleaned_buffers/")
PARQ_PATH = EXP_ROOT / "20220824-0003_clean.parquet"

In [ ]:
df = pd.read_parquet(PARQ_PATH)
df.columns = df.columns.astype("int16")
df = df.T

## Finer DC Adjustments

This section allows us to adjust for DC offset even further if the one provided before was not suitable. Please change this value until the final result is desired. We suggest starting in step size in order of $0.001$ or less to see how it affects the data in the [Visualizations](#Visualizations) section.

In [ ]:
dc_offset = -0.00144
df_offset = df - dc_offset

## Smoothing
This section is currently unused until we determine if smoothing gives us a better FoM value.

**TODO:** Test out filtering

## Normalization

Currently this sections is removed as it does not do anything. In previous versions, `df_norm` was used to find peaks in [Peak Finding](#Peak-Finding), but this posed problems when dealing with noiser data; i.e. this step exaggerates them.

In [ ]:
df_processed = df_offset

# Pulse Shape Discrimination

## Peak Finding
Since we already found peaks in the `data_cleaning.ipynb` notebook, we can just reuse the settings to determine the same values! In a more complete notebook, we can just reuse these values instead of needing to recompute any information using `scipy.signal.find_peaks()`

In [ ]:
SETTINGS_PATH = EXP_ROOT / "settings.toml"

with open(SETTINGS_PATH, "rb") as f:
    exp_info = tomli.load(f)

multipeak_filter_settings = exp_info["multipeak_filter_settings"]
height = multipeak_filter_settings["height"]
prominence = multipeak_filter_settings["prominence"]

output = df_processed.apply(lambda x: signal.find_peaks(x, height=height, prominence=prominence))
peak_idx, props = output.iloc[0,:], output.iloc[1,:]

### Visualize Peak Finding

Use the interactable plot below to decide where the user `peak_offset` and `tail_onset` should lie, these are highlighted in green.

In [ ]:
random_sample = df_processed.sample(10, axis=1, random_state=42)
sample_ids = random_sample.columns   

signal_id_dropdown = widgets.Dropdown(options=sample_ids) 
tail_onset_box = widgets.BoundedIntText(value=8, min=1, max=30)
peak_offset_box = widgets.BoundedIntText(value=20, min=1, max=30)

def plot_peak_finding_interactable(
    signal_id, 
    peak_offset, 
    tail_onset
) -> tuple[list, list]:
   
    output = df_processed.apply(lambda x: get_bases(x, peak_idx[signal_id][0], peak_offset, tail_onset))
    left_bases, right_bases = output.iloc[0,:], output.iloc[1,:]

    plot_bases(
        df_processed.get(signal_id), 
        peak_idx[signal_id][0],
        props[signal_id]["peak_heights"],
        props[signal_id]["left_bases"],  # from scipy.signal.find_peaks()
        left_bases[signal_id],  # from us
        props[signal_id]["right_bases"],  # from scipy.signal.find_peaks()
        right_bases[signal_id],  # from us
    )

    return left_bases, right_bases

In [ ]:
peak_plot_interactable = interactive(
    lambda signal_id, peak_offset, tail_onset: plot_peak_finding_interactable(signal_id, peak_offset, tail_onset),
    signal_id = signal_id_dropdown, 
    peak_offset = peak_offset_box, 
    tail_onset = tail_onset_box
)

display(peak_plot_interactable)

## Integration

**ASSUMPTION:** We will take our `peak_offset` and `tail_onset` result for this integration.

In [ ]:
start_time = time.perf_counter()

peak_offset, tail_onset = peak_offset_box.value, tail_onset_box.value

output = df_processed.apply(lambda x: get_bases(x, peak_idx[x.name][0], peak_offset, tail_onset))
left_bases, right_bases = output.iloc[0,:], output.iloc[1,:]

psd_report = generate_psd(df_processed, left_bases, right_bases, df_processed.max())

print(
    f"Generated PSD report in [\x1b[1;32m{(time.perf_counter() - start_time)*1000:.2f} ms\x1b[0m]."
)

In [ ]:
psd_report

## Visualizations

Here are a couple of key figures used for pulse shape discrimination. If you would not like to run through the integration steps before running the graphs, consider using the [interactive plot](#Interactive) at the end of the notebook for a faster workflow. However, you may find it useful to use this manual section to help you create your figures for publication.

**Dev Note:** I highly recommend the interactive plot since we will be using it's values for the FoM creation! 😬 

### Manual Plotting

#### Tail ($Q_\text{tail}$) vs Total ($Q_\text{total}$)

In [ ]:
fig1, ax1 = plt.subplots(figsize=(12,9))

ax1.scatter(
    psd_report.loc["q_total"],
    psd_report.loc["q_tail"],
    marker=".",
    s=1
)

ax1.set_ylim(-0.4, 5)
ax1.set_xlabel("total integral (a.u.)")
ax1.set_ylabel("tail integral (a.u.)")

plt.show()

#### $Q_\text{tail/total}$ vs Peak Amplitude

In [ ]:
fig2, ax2 = plt.subplots(figsize=(12,9))

ax2.scatter(
    psd_report.loc["amplitude"],
    psd_report.loc["q_tail_total"],
    marker=".",
    s=1
)

ax2.set_ylim(-0.2, 0.5)

ax2.set_xlabel("pulse amplitude ($V$)")
ax2.set_ylabel("tail / total (a.u.)")


plt.show()

In [ ]:
fig3, ax3 = plt.subplots(figsize=(12,9))

n_bins = 100  # If it doesn't show as a nice plot, try increasing resolution (hint: start at 10,000)

ax3.hist(
    psd_report.loc["q_tail_total"],
    bins=n_bins,
    linewidth=2,
    histtype='step',
    orientation='horizontal'
)

ax3.text(
    x=0.75,
    y=0.95,
    s=f"n_bins = {n_bins}",
    transform=ax3.transAxes
)

ax3.set_ylim(-0.2, 0.5)

ax3.set_xlabel("counts")
ax3.set_ylabel("tail / total (a.u.)")

plt.show()

#### <span style="color:#FF9900">Figure of Merit</span>

In [ ]:
fig, ax = plot_fom(                
                    psd_report.loc["q_tail_total"], 
                    n_bins, 
                    mu1=0.11, sigma1=0.04, A1=1000,
                    mu2=0.001, sigma2=0.001, A2=0,
                    unimodal = True
                  )


### Interactive

The following interactive plot runs the whole notebook in a condensed section. Plotting functions are kept here for cleanliness. Please resize `xlim` and `ylim` if the graphs are out of bounds. 

**NOTE:** You find it helpful to comment out where these limits are set to help you visualize the entire plot and show the outliers.

#### Functions

In [ ]:
# ALL IN ONE INTERACTIVE PLOT
def plot_psd(psd_report: pd.DataFrame, n_bins:int = 100) -> tuple[plt.Figure, plt.Axes]:
    label_fs = 14
    
    fig, axs = plt.subplots(1, 3, figsize=(18,6))
    
    # Q_tail vs Q_tot
    axs[0].scatter(
        psd_report.loc["q_total"],
        psd_report.loc["q_tail"],
        marker='.',
        s=1
    )
    
    axs[0].set_ylim(-0.2, 4)
    
    axs[0].set_xlabel("total integral (a.u.)", fontsize=label_fs)
    axs[0].set_ylabel("tail integral (a.u.)", fontsize=label_fs)

    
    hist_y_lims = [-0.1, 0.4]
    
    # Q_tot vs Amplitude
    axs[1].scatter(
        psd_report.loc['amplitude'],
        psd_report.loc["q_tail_total"],
        marker='.',
        s=1
    )
    
    
    axs[1].set_ylim(hist_y_lims)
                    
    axs[1].set_xlabel("pulse amplitude (V)", fontsize=label_fs)
    axs[1].set_ylabel("tail / total (a.u.)", fontsize=label_fs)
    
    
    # Q distribution
    axs[2].hist(
        psd_report.loc["q_tail_total"],
        bins=n_bins,
        linewidth=2,
        histtype='step',
        orientation='horizontal'
    )

    axs[2].text(
        x=0.75,
        y=0.95,
        s=f"n_bins = {n_bins}",
        transform=axs[2].transAxes
    )

    axs[2].set_ylim(hist_y_lims)

    axs[2].set_xlabel("counts", fontsize=label_fs)
    axs[2].set_ylabel("tail / total (a.u.)", fontsize=label_fs)
    
    
    fig.set_facecolor('white')
    fig.tight_layout()
    
    return fig, axs
    
def timed_plot_psd(
    df: pd.DataFrame, 
    dc_offset,
    peak_offset, 
    tail_onset,
    n_bins
) -> dict[tuple[plt.Figure, plt.Axes], pd.DataFrame, dict]:
    start_time = time.perf_counter()

    psd_report = df_to_psd(df, dc_offset, peak_offset, tail_onset)
    
    fig, axs = plot_psd(psd_report, n_bins)

    print(
        f"Action completed in [\x1b[1;32m{(time.perf_counter() - start_time)*1000:.2f} ms\x1b[0m]."
    )
    
    return_dict = {
        "plot": (fig, axs), 
        "psd_report": psd_report, 
        "configs": {
            "dc_offset": dc_offset,
            "peak_offset": peak_offset,
            "tail_onset": tail_onset
        }
    }
    
    return return_dict

#### Plotting


**PROPOSED FEATURE:** be able to adjust xlim / ylim without replotting?

In [ ]:
tail_onset_box_end = widgets.IntText(value=8, description="Tail Onset")
peak_offset_box_end = widgets.IntText(value=20, description="Peak Offset")
dc_offset_box = widgets.FloatText(value=-0.00144, description="DC Offset")
n_bins_box = widgets.IntText(value=100, description="n_bins")

ui_full_plot = widgets.VBox(
    [
        widgets.HBox([dc_offset_box, n_bins_box]),
        widgets.HBox([tail_onset_box_end, peak_offset_box_end])
    ]
)

interactive_plot_full = interactive(
    lambda dc_offset, peak_offset, tail_onset, n_bins:
    timed_plot_psd(
        df, 
        dc_offset,
        peak_offset, 
        tail_onset, 
        n_bins
    ),
    dc_offset= dc_offset_box,
    peak_offset= peak_offset_box_end,
    tail_onset= tail_onset_box_end,
    n_bins= n_bins_box,
)

display(interactive_plot_full) 

## Report

Here we generate a plot to show the Figure of Merit (FoM) calculated using the Full Widths at Half Maximum (FWHM). These are defined as follows:

$$\text{FoM} = \dfrac{\mu_2 - \mu_1}{\text{FWHM}(\sigma_1) + \text{FWHM}(\sigma_2)}$$


$$\text{FWHM}(\sigma) = 2\sqrt{2 \ln 2} \cdot \sigma$$

where, $\mu_i$, $\sigma_i$ are the mean and standard deviation of each Gaussian distribution, respectively.

**Note:** for this section, we will be using the result obtained from the interactive plot obtained prior. Ensure that interactive plot is what you expect the distribution to look like.

### Functions & Widgets

In [ ]:
gauss1_settings = widgets.HBox(
    [
        widgets.FloatText(value=0.12, description="$\mu_1$"),
        widgets.FloatText(value=0.04, description="$\sigma_1$"),
        widgets.FloatText(value=1000, description="$A_1$")
    ]
)

gauss2_settings = widgets.HBox(
    [
        widgets.FloatText(value=0.001, description="$\mu_2$"),
        widgets.FloatText(value=0.001, description="$\sigma_2$"),
        widgets.FloatText(value=0, description="$A_2$")
    ]
)

unimodal_checkbox = widgets.Checkbox(value=True, description="Unimodal")
fom_widgets = widgets.VBox([unimodal_checkbox, gauss1_settings, gauss2_settings])

### Plotting
Here we generate the actual plot, adjust $\mu_i, \sigma_i, A_i$ to try and fit the plots better.

In [ ]:
args_map_fom = {
    "mu1": gauss1_settings.children[0],
    "sigma1": gauss1_settings.children[1],
    "A1": gauss1_settings.children[2],
    "mu2": gauss2_settings.children[0],
    "sigma2": gauss2_settings.children[1],
    "A2": gauss2_settings.children[2],
    "unimodal": unimodal_checkbox
}

fom_interactive = interactive_output(
    lambda mu1, sigma1, A1, mu2, sigma2, A2, unimodal: 
    plot_fom(
        interactive_plot_full.result["psd_report"].loc["q_tail_total"],
        n_bins_box.value,
        mu1, sigma1, A1,
        mu2, sigma2, A2,
        unimodal
    ), 
    args_map_fom
)

display(fom_widgets, fom_interactive)

In [ ]:
def gen_msa(gauss_settings):
    msa = [child.value for child in gauss_settings.children]
    return {"mu": msa[0], "sigma": msa[1], "A": msa[2]}

gauss1_params = gen_msa(gauss1_settings)
gauss2_params = gen_msa(gauss2_settings)

In [ ]:
gauss_settings_df = pd.DataFrame([gauss1_params, gauss2_params], index=["1", "2"])
gauss_settings_df